# 03 — Dataset Preprocessing (Crops + DenseNet121 Input)

**Skripsi:** Klasifikasi Tumor Paru pada Citra CT Scan Menggunakan DenseNet121
dan Explainable Artificial Intelligence berbasis Grad-CAM.

Notebook ini adalah **orkestrasi** tahap preprocessing dataset penelitian:
memeta manifest ke DICOM, mendokumentasikan keputusan context margin,
menjalankan batch preprocessing, lalu quality control.

Pekerjaan batch dipisahkan ke `scripts/06_preprocess_dataset.py` agar tidak
berat di notebook dan bisa dijalankan ulang dari terminal.

### Batas kerja

Notebook ini **tidak** melakukan:

- training DenseNet121
- Grad-CAM
- perhitungan Accuracy/Precision/Recall/F1/ROC-AUC
- split baru (Fold dari `nodule_manifest.csv` dipakai apa adanya)
- oversampling / undersampling / augmentation
- perubahan threshold label

### Rantai preprocessing (identik dengan `01_preprocessing.ipynb`)

```
DICOM -> HU -> lung window [-1350, 150] -> [0,1]
      -> crop bbox + margin 5 mm (per-axis, ceil)
      -> resize 224x224                 -> dataset/03_crops/*.png
      -> grayscale -> 3 kanal HWC (224,224,3)
      -> densenet.preprocess_input      -> dataset/04_preprocessed/*.npy
```

## 1. Configuration

Path diturunkan dari `PROJECT_ROOT` agar portabel antar mesin.

In [1]:
# --- CELL 1 - KONFIGURASI ------------------------------------------------
import importlib.util
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

# --- parameter preprocessing (TERKUNCI, sama dengan notebook 01) ---------
WINDOW_HU_MIN = -1350.0
WINDOW_HU_MAX = 150.0
TARGET_SIZE = (224, 224)
NUM_CHANNELS = 3
CHANNEL_LAYOUT = "channels_last"
BBOX_XY_CONVENTION = "transposed"           # BBox_X = baris, BBox_Y = kolom
SLICE_SELECTION = "geometric_lower_middle"   # (n-1)//2

# --- MARGIN FINAL (hasil audit empiris, disetujui penulis skripsi) --------
# Bukan placeholder. Dipilih dari data, BUKAN dari performa model.
MARGIN_MM_FINAL = 5.0
MARGIN_DECISION_REASON = (
    "Nodul median 45% dari luas crop; upsample median x7.7 ke 224; hanya 5.9% "
    "sample butuh >=10x upsample (dibanding 47.7% pada 3 mm dan 79.0% pada "
    "0 mm). Konteks parenkim dan pembuluh darah sekitar nodul tetap terlihat. "
    "Clipping 0% karena jarak minimum bbox ke tepi FOV 21.9 mm."
)
MARGIN_CANDIDATES_MM = (0.0, 3.0, 5.0, 8.0)
MARGIN_CANDIDATES_PX = (0, 16, 32)

# --- path -----------------------------------------------------------------
MANIFEST_RELPATH = Path("dataset/05_splits/nodule_manifest.csv")
RAW_RELDIR = Path("dataset/01_raw")
CROP_RELDIR = Path("dataset/03_crops")
PREP_RELDIR = Path("dataset/04_preprocessed")
RESULTS_RELDIR = Path("results/preprocessing_qc")
AUDIT_SCRIPT_RELPATH = Path("scripts/04_validate_preprocessing.py")
BATCH_SCRIPT_RELPATH = Path("scripts/06_preprocess_dataset.py")


def find_project_root(marker: Path, start: Path | None = None) -> Path:
    base = Path(start).resolve() if start else Path.cwd().resolve()
    for candidate in [base, *base.parents]:
        if (candidate / marker).exists():
            return candidate
    searched = "\n".join(f"  - {p}" for p in [base, *base.parents])
    raise FileNotFoundError(
        f"PROJECT_ROOT tidak ditemukan.\n"
        f"Notebook mencari: {marker}\n\n"
        f"Direktori yang diperiksa:\n{searched}"
    )


PROJECT_ROOT = find_project_root(MANIFEST_RELPATH)
MANIFEST_FILE = PROJECT_ROOT / MANIFEST_RELPATH
RAW_DIR = PROJECT_ROOT / RAW_RELDIR
CROP_DIR = PROJECT_ROOT / CROP_RELDIR
PREP_DIR = PROJECT_ROOT / PREP_RELDIR
RESULTS_DIR = PROJECT_ROOT / RESULTS_RELDIR

print("KONFIGURASI")
print("-" * 70)
print(f"Python         : {sys.version.split()[0]}")
print(f"PROJECT_ROOT   : {PROJECT_ROOT}")
print(f"Input manifest : {MANIFEST_RELPATH}")
print(f"Raw DICOM      : {RAW_RELDIR}")
print(f"Output crop    : {CROP_RELDIR}")
print(f"Output preproc : {PREP_RELDIR}")
print(f"Window         : [{WINDOW_HU_MIN}, {WINDOW_HU_MAX}] HU  (TERKUNCI)")
print(f"Margin final   : {MARGIN_MM_FINAL} mm  (TERKUNCI, per-axis)")
print(f"Konvensi bbox  : {BBOX_XY_CONVENTION}")
print(f"Sample         : {SLICE_SELECTION} -> (n-1)//2")
print(f"Channel layout : {CHANNEL_LAYOUT} {TARGET_SIZE} x {NUM_CHANNELS}")


KONFIGURASI
----------------------------------------------------------------------
Python         : 3.12.10
PROJECT_ROOT   : C:\230016_rivan\source-code2020
Input manifest : dataset\05_splits\nodule_manifest.csv
Raw DICOM      : dataset\01_raw
Output crop    : dataset\03_crops
Output preproc : dataset\04_preprocessed
Window         : [-1350.0, 150.0] HU  (TERKUNCI)
Margin final   : 5.0 mm  (TERKUNCI, per-axis)
Konvensi bbox  : transposed
Sample         : geometric_lower_middle -> (n-1)//2
Channel layout : channels_last (224, 224) x 3


In [2]:
# --- CELL 1.1 - MUAT MODUL AUDIT -----------------------------------------
def load_module(name: str, relpath: Path):
    """Impor script project sebagai modul, agar pipeline tidak diduplikasi."""
    path = PROJECT_ROOT / relpath
    if not path.is_file():
        raise FileNotFoundError(f"Script tidak ditemukan: {relpath}")
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod


audit = load_module("lidc_audit_v04", AUDIT_SCRIPT_RELPATH)
batch = load_module("lidc_batch_v06", BATCH_SCRIPT_RELPATH)

print("Modul dimuat:")
print(f"  {AUDIT_SCRIPT_RELPATH.name:<34} "
      f"({len([n for n in dir(audit) if not n.startswith('_')])} nama publik)")
print(f"  {BATCH_SCRIPT_RELPATH.name:<34} "
      f"({len([n for n in dir(batch) if not n.startswith('_')])} nama publik)")
print("\nFungsi preprocessing dipakai dari modul audit, bukan ditulis ulang:")
for fn in ("to_hounsfield", "apply_window", "resize_to", "to_three_channel",
           "margin_px_pair_from_mm", "margin_px_from_mm", "build_crop_box",
           "middle_index"):
    print(f"  - {fn}")


Modul dimuat:
  04_validate_preprocessing.py       (53 nama publik)
  06_preprocess_dataset.py           (38 nama publik)

Fungsi preprocessing dipakai dari modul audit, bukan ditulis ulang:
  - to_hounsfield
  - apply_window
  - resize_to
  - to_three_channel
  - margin_px_pair_from_mm
  - margin_px_from_mm
  - build_crop_box
  - middle_index


## 2. Load Manifest

Daftar sample final berasal dari notebook 02. **Fold tidak diubah.**

In [3]:
# --- CELL 2 - MANIFEST SAMPEL --------------------------------------------
df = batch.load_manifest(MANIFEST_FILE)

print(f"Baris            : {len(df)}")
print(f"PatientID unik   : {df['PatientID'].nunique()}")
print(f"SeriesUID unik   : {df['SeriesInstanceUID'].nunique()}")
print(f"NoduleClusterID  : {df['NoduleClusterID'].nunique()}")
print(f"Label            : {df['Label'].value_counts().to_dict()}")
print(f"Fold             : {sorted(df['Fold'].unique().tolist())}")
print(f"Sample per fold  : {df['Fold'].value_counts().sort_index().to_dict()}")

print("\nDistribusi label per fold:")
display(pd.crosstab(df["Fold"], df["Label"]))


Baris            : 524
PatientID unik   : 225
SeriesUID unik   : 225
NoduleClusterID  : 524
Label            : {'Benign': 381, 'Malignant': 143}
Fold             : [1, 2, 3, 4, 5]
Sample per fold  : {1: 82, 2: 126, 3: 106, 4: 97, 5: 113}

Distribusi label per fold:


Label,Benign,Malignant
Fold,,
1,57,25
2,94,32
3,74,32
4,69,28
5,87,26


## 3. Validasi Mapping Manifest → DICOM

Setiap baris manifest harus dapat dipetakan ke DICOM: `PatientID`,
`SeriesInstanceUID`, `NoduleClusterID`, bounding box, dan `BBox_Z_Slices`.

Bila ada sample yang tidak terpetakan, proses **dihentikan** dan masalahnya
ditampilkan. Sample tidak pernah dibuang diam-diam.

In [4]:
# --- CELL 3 - INDEX DICOM + VALIDASI MAPPING -----------------------------
t_index = pd.Timestamp.now()
series_index = batch.index_dicom(RAW_DIR)
elapsed = (pd.Timestamp.now() - t_index).total_seconds()

n_dcm = sum(len(v) for v in series_index.values())
print(f"Folder DICOM       : {RAW_RELDIR}")
print(f"Seri ditemukan     : {len(series_index)}")
print(f"Total file DICOM   : {n_dcm:,}")
print(f"Waktu index        : {elapsed:.0f}s")

print("\nVALIDASI MAPPING")
print("-" * 70)
problems = batch.validate_mapping(df, series_index)
if len(problems):
    print(problems.to_string(index=False))
    raise ValueError(
        f"{len(problems)} sample tidak dapat dipetakan ke DICOM.\n"
        f"PROSES DIHENTIKAN. Sample TIDAK dibuang. Perbaiki sumber data."
    )

found = df["SeriesInstanceUID"].isin(series_index).sum()
print(f"  sample manifest        : {len(df)}")
print(f"  sample DICOM ditemukan : {found}")
print(f"  sample gagal           : {len(df) - found}")
print(f"  seri unik dipakai      : {df['SeriesInstanceUID'].nunique()}")
print(f"  pasien unik            : {df['PatientID'].nunique()}")
print("\nSemua sample terpetakan. Lanjut ke validasi koordinat.")


Folder DICOM       : dataset\01_raw
Seri ditemukan     : 306
Total file DICOM   : 76,661
Waktu index        : 26s

VALIDASI MAPPING
----------------------------------------------------------------------
  sample manifest        : 524
  sample DICOM ditemukan : 524
  sample gagal           : 0
  seri unik dipakai      : 225
  pasien unik            : 225

Semua sample terpetakan. Lanjut ke validasi koordinat.


## 4. Validasi Koordinat

Konvensi yang dipakai: **`BBox_X` = baris (row)**, **`BBox_Y` = kolom (column)**.
Konvensi ini sudah divalidasi pada `01_preprocessing.ipynb` dan **tidak
diubah** di sini.

Slice dipilih secara geometris (`(n-1)//2`), bukan berbasis intensitas.

In [5]:
# --- CELL 4 - SANITY CHECK KOORDINAT -------------------------------------
import random

rng = np.random.default_rng(42)
random.seed(42)

uids = sorted(df["SeriesInstanceUID"].unique())
check_uids = list(rng.choice(uids, size=10, replace=False))

rows = []
for uid in check_uids:
    s = audit.CtSeries(uid, series_index[uid])
    g = df[df["SeriesInstanceUID"] == uid]
    for _, r in g.iterrows():
        box = audit.build_crop_box(r, s, 0, 0)
        z0 = max(0, int(r["BBox_Z_Min"]))
        z1 = min(s.n_slices - 1, int(r["BBox_Z_Max"]))
        k = z0 + audit.middle_index(z1 - z0 + 1)
        hu = s.read_hu(k)
        crop = hu[box["r0"]:box["r1"] + 1, box["c0"]:box["c1"] + 1]
        rows.append({
            "NoduleClusterID": str(r["NoduleClusterID"])[-12:],
            "rows_x_cols": f"{s.n_rows}x{s.n_cols}",
            "bbox_in_range": bool(box["core_r0"] >= 0
                                  and box["core_r1"] < s.n_rows
                                  and box["core_c0"] >= 0
                                  and box["core_c1"] < s.n_cols),
            "crop_shape": f"{crop.shape[0]}x{crop.shape[1]}",
            "crop_kosong": bool(crop.size == 0),
            "n_slices_z": z1 - z0 + 1,
            "k_abs": k,
            "k_valid": bool(0 <= k < s.n_slices),
            "HU_min": float(crop.min()) if crop.size else float("nan"),
            "HU_max": float(crop.max()) if crop.size else float("nan"),
            "HU_wajar": bool(crop.size and -1200 <= crop.min()
                             and crop.max() <= 3000),
        })

coord = pd.DataFrame(rows)
print("SANITY CHECK KOORDINAT (10 seri, konvensi BBox_X=row)")
print("-" * 70)
print(coord.to_string(index=False))

print("\nRINGKASAN")
print(f"  bbox dalam batas slice : {coord['bbox_in_range'].all()}")
print(f"  crop tidak kosong      : {not coord['crop_kosong'].any()}")
print(f"  k valid                : {coord['k_valid'].all()}")
print(f"  HU masuk akal          : {coord['HU_wajar'].all()}")
print(f"  n_slices_z = BBox_Z_Slices : dicek lewat manifest, "
      f"nilai pada manifest: {sorted(df['BBox_Z_Slices'].unique())[:8]} ...")
assert coord["bbox_in_range"].all(), "bbox di luar batas slice"
assert not coord["crop_kosong"].any(), "ada crop kosong"
assert coord["k_valid"].all(), "indeks slice di luar rentang"
assert coord["HU_wajar"].all(), "HU di luar rentang masuk akal"
print("\nKoordinat konsisten. Lanjut ke audit margin.")


SANITY CHECK KOORDINAT (10 seri, konvensi BBox_X=row)
----------------------------------------------------------------------
NoduleClusterID rows_x_cols  bbox_in_range crop_shape  crop_kosong  n_slices_z  k_abs  k_valid  HU_min  HU_max  HU_wajar
   6790879_N001     512x512           True      13x13        False           3     86     True  -870.0     3.0      True
   2922488_N001     512x512           True        6x8        False           2     52     True  -842.0   -55.0      True
   2922488_N002     512x512           True      14x13        False           4     60     True  -863.0    30.0      True
   8320084_N001     512x512           True      14x18        False           8     83     True  -952.0    18.0      True
   8320084_N002     512x512           True      68x84        False          31    236     True -1024.0   748.0      True
   6834230_N001     512x512           True        8x6        False           3     38     True  -814.0    29.0      True
   6834230_N002     512x512 

## 5. Audit Context Margin

Context margin sebelumnya masih placeholder `0`. Angka itu **bukan**
keputusan metodologi.

Audit di bawah memakai fungsi dari `scripts/04_validate_preprocessing.py`
(`build_crop_box`, `margin_px_pair_from_mm`) dan dijalankan pada **524 sample
final**, bukan satu-dua contoh.

In [6]:
# --- CELL 5 - AUDIT MARGIN PADA 524 SAMPLE --------------------------------
# Margin mm dikonversi per-axis: ceil(mm / row_spacing) dan ceil(mm /
# col_spacing). Rata-rata spacing TIDAK dipakai.
margin_rows = []
t_audit = pd.Timestamp.now()

for uid, g in df.groupby("SeriesInstanceUID", sort=False):
    s = audit.CtSeries(uid, series_index[uid])
    for _, r in g.iterrows():
        for mm in MARGIN_CANDIDATES_MM:
            mr, mc = audit.margin_px_pair_from_mm(mm, s.row_spacing,
                                                  s.col_spacing)
            box = audit.build_crop_box(r, s, mr, mc)
            core_px = max(box["core_h"], box["core_w"])
            crop_px = max(box["crop_h"], box["crop_w"])
            margin_rows.append({
                "Margin": f"{mm:g} mm", "Jenis": "mm", "Nilai": mm,
                "NoduleClusterID": str(r["NoduleClusterID"]),
                "PatientID": r["PatientID"],
                "SeriesInstanceUID": uid,
                "margin_row_px": mr, "margin_col_px": mc,
                "bbox_valid": bool(box["core_r1"] >= box["core_r0"]
                                   and box["core_c1"] >= box["core_c0"]),
                "crop_h_px": box["crop_h"], "crop_w_px": box["crop_w"],
                "crop_h_mm": round(box["crop_h_mm"], 1),
                "crop_w_mm": round(box["crop_w_mm"], 1),
                "core_max_mm": round(max(box["core_h"] * s.row_spacing,
                                         box["core_w"] * s.col_spacing), 1),
                "nodule_over_crop_pct": round(core_px / crop_px * 100, 1),
                "upsample": round(max(TARGET_SIZE) / crop_px, 1),
                "clipped": bool(box["clipped_row"] or box["clipped_col"]),
                "margin_row_mm": round(box["actual_margin_row_mm"], 2),
                "margin_col_mm": round(box["actual_margin_col_mm"], 2),
                "di_luar_FOV": bool(box["core_r0"] < 0 or box["core_r1"] >= s.n_rows
                                    or box["core_c0"] < 0
                                    or box["core_c1"] >= s.n_cols),
                "terlalu_kecil": bool(crop_px < 32),
                "terlalu_besar": bool(crop_px > 200),
            })
        for p in MARGIN_CANDIDATES_PX:
            box = audit.build_crop_box(r, s, p, p)
            crop_px = max(box["crop_h"], box["crop_w"])
            core_px = max(box["core_h"], box["core_w"])
            margin_rows.append({
                "Margin": f"{p} px", "Jenis": "px", "Nilai": float(p),
                "NoduleClusterID": str(r["NoduleClusterID"]),
                "PatientID": r["PatientID"],
                "SeriesInstanceUID": uid,
                "margin_row_px": p, "margin_col_px": p,
                "bbox_valid": True,
                "crop_h_px": box["crop_h"], "crop_w_px": box["crop_w"],
                "crop_h_mm": round(box["crop_h_mm"], 1),
                "crop_w_mm": round(box["crop_w_mm"], 1),
                "core_max_mm": round(max(box["core_h"] * s.row_spacing,
                                         box["core_w"] * s.col_spacing), 1),
                "nodule_over_crop_pct": round(core_px / crop_px * 100, 1),
                "upsample": round(max(TARGET_SIZE) / crop_px, 1),
                "clipped": bool(box["clipped_row"] or box["clipped_col"]),
                "margin_row_mm": round(box["actual_margin_row_mm"], 2),
                "margin_col_mm": round(box["actual_margin_col_mm"], 2),
                "di_luar_FOV": False,
                "terlalu_kecil": bool(crop_px < 32),
                "terlalu_besar": bool(crop_px > 200),
            })

margin_audit = pd.DataFrame(margin_rows)
audit_s = (pd.Timestamp.now() - t_audit).total_seconds()
print(f"Audit margin pada {len(df)} sample x "
      f"{len(MARGIN_CANDIDATES_MM) + len(MARGIN_CANDIDATES_PX)} kandidat "
      f"= {len(margin_audit)} baris ({audit_s:.0f}s)")


Audit margin pada 524 sample x 7 kandidat = 3668 baris (24s)


In [7]:
# --- CELL 5.1 - TABEL PERBANDINGAN KANDIDAT MARGIN -----------------------
order = ([f"{m:g} mm" for m in MARGIN_CANDIDATES_MM]
         + [f"{p} px" for p in MARGIN_CANDIDATES_PX])

cmp_rows = []
for m in order:
    g = margin_audit[margin_audit["Margin"] == m]
    cmp_rows.append({
        "Margin": m, "Jenis": g["Jenis"].iloc[0],
        "row_px": int(g["margin_row_px"].iloc[0]),
        "col_px": int(g["margin_col_px"].iloc[0]),
        "bbox_valid": f"{g['bbox_valid'].mean() * 100:.1f}%",
        "di_luar_FOV": f"{g['di_luar_FOV'].mean() * 100:.1f}%",
        "ter-clipping": f"{g['clipped'].mean() * 100:.1f}%",
        "crop_med_px": f"{g['crop_h_px'].median():.0f}x{g['crop_w_px'].median():.0f}",
        "crop_med_mm": f"{g['crop_h_mm'].median():.1f}x{g['crop_w_mm'].median():.1f}",
        "nodul_med_mm": f"{g['core_max_mm'].median():.1f}",
        "nodul/crop": f"{g['nodule_over_crop_pct'].median():.1f}%",
        "upsample_med": f"{g['upsample'].median():.1f}",
        "upsample_p90": f"{g['upsample'].quantile(0.90):.1f}",
        ">=10x upsample": f"{(g['upsample'] >= 10).mean() * 100:.1f}%",
        "<32px": f"{g['terlalu_kecil'].mean() * 100:.1f}%",
        ">200px": f"{g['terlalu_besar'].mean() * 100:.1f}%",
    })
margin_compare = pd.DataFrame(cmp_rows)

print("PERBANDINGAN KANDIDAT CONTEXT MARGIN (524 sample)")
print("=" * 118)
print(margin_compare.to_string(index=False))

print("\nRINGKASAN PER SERI (kandidat utama, 5 seri pertama):")
sub = margin_audit[(margin_audit["Margin"] == f"{MARGIN_MM_FINAL:g} mm")]
by_ser = (sub.groupby("SeriesInstanceUID")
          .agg(n_cluster=("NoduleClusterID", "size"),
               crop_med_px=("crop_h_px", "median"),
               upsample_med=("upsample", "median"),
               nodul_crop=("nodule_over_crop_pct", "median"))
          .head(5))
print(by_ser.to_string())


PERBANDINGAN KANDIDAT CONTEXT MARGIN (524 sample)
Margin Jenis  row_px  col_px bbox_valid di_luar_FOV ter-clipping crop_med_px crop_med_mm nodul_med_mm nodul/crop upsample_med upsample_p90 >=10x upsample <32px >200px
  0 mm    mm       0       0     100.0%        0.0%         0.0%       12x12     8.4x8.4          8.9     100.0%         17.2         28.0          79.0% 87.2%   0.0%
  3 mm    mm       5       5     100.0%        0.0%         0.0%       22x22   14.9x15.0          8.9      57.4%          9.7         12.4          47.7% 77.5%   0.0%
  5 mm    mm       8       8     100.0%        0.0%         0.0%       28x28   19.0x19.0          8.9      45.5%          7.7          9.6           5.9% 62.8%   0.0%
  8 mm    mm      12      12     100.0%        0.0%         0.0%       37x37   25.1x25.1          8.9      35.0%          5.9          7.1           0.0% 10.1%   0.0%
  0 px    px       0       0     100.0%        0.0%         0.0%       12x12     8.4x8.4          8.9     100.0%   

In [8]:
# --- CELL 5.2 - ATURAN PEMILIHAN MARGIN ----------------------------------
print("ATURAN PEMILIHAN MARGIN (bukan berbasis performa model)")
print("=" * 72)
print("Model BELUM dilatih, jadi margin TIDAK boleh dipilih dari accuracy,")
print("AUC, atau performa validasi. Kriteria yang dipakai:")
print()
print("  1. Mempertahankan konteks jaringan sekitar nodule")
print("     -> nodul masih mengisi sebagian besar crop, konteks parenkim terlihat")
print("  2. Tidak terlalu banyak background tidak relevan")
print("     -> nodul tidak terlalu kecil dibanding frame")
print("  3. Clipping akibat batas FOV tetap terkendali")
print("     -> jarak minimum bbox ke tepi FOV diperiksa")
print("  4. Ukuran crop cukup untuk resize ke 224x224")
print("     -> memperkecil faktor upsample yang terlalu besar")
print("  5. Konsisten diterapkan ke seluruh dataset")
print("     -> konversi per-axis otomatis pada semua sample")
print()
print(f"MARGIN FINAL : {MARGIN_MM_FINAL} mm")
print(f"ALASAN        : {MARGIN_DECISION_REASON}")
print()
print("TIDAK ada model yang dilatih pada tahap ini. Angka di atas purely")
print("berasal dari audit geometris/preprocessing pada 524 sample.")


ATURAN PEMILIHAN MARGIN (bukan berbasis performa model)
Model BELUM dilatih, jadi margin TIDAK boleh dipilih dari accuracy,
AUC, atau performa validasi. Kriteria yang dipakai:

  1. Mempertahankan konteks jaringan sekitar nodule
     -> nodul masih mengisi sebagian besar crop, konteks parenkim terlihat
  2. Tidak terlalu banyak background tidak relevan
     -> nodul tidak terlalu kecil dibanding frame
  3. Clipping akibat batas FOV tetap terkendali
     -> jarak minimum bbox ke tepi FOV diperiksa
  4. Ukuran crop cukup untuk resize ke 224x224
     -> memperkecil faktor upsample yang terlalu besar
  5. Konsisten diterapkan ke seluruh dataset
     -> konversi per-axis otomatis pada semua sample

MARGIN FINAL : 5.0 mm
ALASAN        : Nodul median 45% dari luas crop; upsample median x7.7 ke 224; hanya 5.9% sample butuh >=10x upsample (dibanding 47.7% pada 3 mm dan 79.0% pada 0 mm). Konteks parenkim dan pembuluh darah sekitar nodul tetap terlihat. Clipping 0% karena jarak minimum bbox ke te

## 6. Generate Final Crops + DenseNet121 Input

Menjalankan `scripts/06_preprocess_dataset.py`.

### KPrevention preprocessing ganda

File di `dataset/04_preprocessed/` **sudah** berupa output
`tf.keras.applications.densenet.preprocess_input`. Nilai itu harus diberikan
langsung ke model — **jangan** memanggil `preprocess_input` lagi.

**Format penyimpanan:**

| Direktori | Format | Isi |
|---|---|---|
| `dataset/03_crops/` | PNG uint8 grayscale | citra windowed `[0,1]` → 0..255, untuk inspeksi visual |
| `dataset/04_preprocessed/` | `.npy` float32 | **output `preprocess_input`**, siap pakai training |

`.npy` dipilih karena menyimpan float32 apa adanya (tanpa kuantisasi),
tidak perlu decode gambar, dan tidak menguji ulang preprocessing.

In [9]:
# --- CELL 6 - JALANKAN BATCH PREPROCESSING -------------------------------
import subprocess

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Menjalankan batch preprocessing...")
print(f"  script : {BATCH_SCRIPT_RELPATH}")
print(f"  margin : {MARGIN_MM_FINAL} mm")
print(f"  python : {sys.executable}\n")

proc = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / BATCH_SCRIPT_RELPATH),
     "--margin-mm", str(MARGIN_MM_FINAL)],
    cwd=str(PROJECT_ROOT), capture_output=True, text=True,
    encoding="utf-8", errors="replace",
)
print(proc.stdout[-6000:])
if proc.returncode != 0:
    print(proc.stderr[-3000:])
    raise RuntimeError(
        f"Batch preprocessing gagal (exit {proc.returncode}). "
        f"Output di atas menunjukkan penyebabnya."
    )
print(f"Batch selesai (exit {proc.returncode}).")


Menjalankan batch preprocessing...
  script : scripts\06_preprocess_dataset.py
  margin : 5.0 mm
  python : c:\230016_rivan\source-code2020\.venv\Scripts\python.exe

BATCH PREPROCESSING LIDC-IDRI
PROJECT_ROOT     : C:\230016_rivan\source-code2020
Manifest         : dataset\05_splits\nodule_manifest.csv
Raw DICOM        : dataset\01_raw
Output crop      : dataset\03_crops
Output preproc   : dataset\04_preprocessed
Window (terkunci): [-1350.0, 150.0] HU
Margin final     : 5.0 mm (per-axis)
Konvensi bbox    : transposed
Channel layout   : channels_last (224, 224) x 3
------------------------------------------------------------------------------
Input            : 524 sample, 225 pasien, 225 seri
Label            : {'Benign': 381, 'Malignant': 143}
Fold             : [1, 2, 3, 4, 5]

DICOM ditemukan  : 306 seri (23s)

VALIDASI MAPPING
  semua 524 sample terpetakan ke DICOM  [OK]
  seri unik dipakai : 225
  pasien unik       : 225

GENERATE (margin 5.0 mm)
  50/524 sample (2s, 0 gagal)
  10

## 7. Quality Control

Pemeriksaan otomatis atas output preprocessing.

In [10]:
# --- CELL 7 - QUALITY CONTROL --------------------------------------------
prep_manifest_file = PREP_DIR / "preprocessed_manifest.csv"
if not prep_manifest_file.is_file():
    raise FileNotFoundError(f"Manifest output tidak ditemukan: {prep_manifest_file}")

man = pd.read_csv(prep_manifest_file, dtype={
    "PatientID": str, "SeriesInstanceUID": str, "NoduleClusterID": str})

qc = pd.read_csv(PREP_DIR / "preprocessing_qc.csv")
print("QUALITY CONTROL")
print("=" * 96)
print(qc.to_string(index=False))
n_bad = int((qc["status"] != "OK").sum())
print(f"\nPemeriksaan GAGAL: {n_bad}")
if n_bad:
    print(qc[qc["status"] != "OK"].to_string(index=False))
    raise AssertionError(f"{n_bad} pemeriksaan QC gagal. Lihat tabel di atas.")
print("Seluruh pemeriksaan QC lolos.")


QUALITY CONTROL
                                        cek status                                            detail
                jumlah input manifest penuh     OK                                               524
         jumlah output = jumlah input penuh     OK                                           524/524
      tidak ada sample duplikat di manifest     OK                                                 0
setiap NoduleClusterID punya tepat 1 output     OK                                                 1
                      semua Fold tetap 1..5     OK                                   [1, 2, 3, 4, 5]
    distribusi label sesuai input (381/143)     OK input {0: 381, 1: 143} -> output {0: 381, 1: 143}
                      LabelBinary hanya 0/1     OK                                            [0, 1]
             tidak ada NaN pada model input     OK                                                 0
             tidak ada Inf pada model input     OK                         

In [11]:
# --- CELL 7.1 - VERIFIKASI ULANG DARI FILE (independen) ------------------
# Membaca ulang file .npy dari disk, bukan mempercayai manifest.
n_checked = 0
bad_shape = bad_dtype = n_nan = n_inf = 0
mins, maxs = [], []
for p in man["PreprocessedPath"]:
    arr = np.load(PROJECT_ROOT / p)
    if arr.shape != (224, 224, 3):
        bad_shape += 1
    if arr.dtype != np.float32:
        bad_dtype += 1
    if np.isnan(arr).any():
        n_nan += 1
    if np.isinf(arr).any():
        n_inf += 1
    mins.append(float(arr.min()))
    maxs.append(float(arr.max()))
    n_checked += 1

print("VERIFIKASI ULANG DARI FILE .npy (semua file dibaca)")
print("-" * 70)
print(f"  file diperiksa            : {n_checked}")
print(f"  shape != (224,224,3)      : {bad_shape}")
print(f"  dtype != float32          : {bad_dtype}")
print(f"  mengandung NaN            : {n_nan}")
print(f"  mengandung Inf            : {n_inf}")
print(f"  rentang nilai min         : {min(mins):.4f}")
print(f"  rentang nilai max         : {max(maxs):.4f}")
print(f"  rentang dalam [-2.5, 2.7] : {min(mins) >= -2.5 and max(maxs) <= 2.7}")
print("\nCatatan kanal: DenseNet preprocess_input memakai mean/std ImageNet")
print("PER KANAL yang berbeda (0.485/0.456/0.406 dan 0.229/0.224/0.225).")
print("Karena itu kanal pada file SUDAH diproses BERBEDA antar kanal. Yang")
print("harus identik adalah input grayscale SEBELUM preprocess_input.")

assert bad_shape == 0, f"{bad_shape} file shape salah"
assert bad_dtype == 0, f"{bad_dtype} file dtype salah"
assert n_nan == 0, f"{n_nan} file mengandung NaN"
assert n_inf == 0, f"{n_inf} file mengandung Inf"
print("\nSemua file valid.")


VERIFIKASI ULANG DARI FILE .npy (semua file dibaca)
----------------------------------------------------------------------
  file diperiksa            : 524
  shape != (224,224,3)      : 0
  dtype != float32          : 0
  mengandung NaN            : 0
  mengandung Inf            : 0
  rentang nilai min         : -1.8788
  rentang nilai max         : 2.6400
  rentang dalam [-2.5, 2.7] : True

Catatan kanal: DenseNet preprocess_input memakai mean/std ImageNet
PER KANAL yang berbeda (0.485/0.456/0.406 dan 0.229/0.224/0.225).
Karena itu kanal pada file SUDAH diproses BERBEDA antar kanal. Yang
harus identik adalah input grayscale SEBELUM preprocess_input.

Semua file valid.


In [12]:
# --- CELL 7.2 - ANTI DOUBLE PREPROCESSING ---------------------------------
import tensorflow as tf

print("UJI ANTI DOUBLE PREPROCESSING")
print("-" * 70)
print("Seandainya file .npy BELUM diproses, maka")
print("  preprocess_input(file * 255) == file  (idempoten).")
print("Karena preprocessing sudah diterapkan, hasilannya harus BERBEDA.\n")

n_idem = 0
for p in man["PreprocessedPath"].head(8):
    arr = np.load(PROJECT_ROOT / p)[None, ...]
    again = np.asarray(
        tf.keras.applications.densenet.preprocess_input(
            (arr * 255.0).copy(), data_format="channels_last"), dtype=np.float32)
    d = float(np.abs(again - arr).max())
    if d <= 1e-6:
        n_idem += 1
print(f"  file idempoten : {n_idem}/8  (harus 0)")
assert n_idem == 0, "Output belum diproses - ada risiko double preprocessing!"
print("\nOutput SUDAH diproses. Saat training, berikan langsung ke model")
print("TANPA memanggil preprocess_input lagi.")


UJI ANTI DOUBLE PREPROCESSING
----------------------------------------------------------------------
Seandainya file .npy BELUM diproses, maka
  preprocess_input(file * 255) == file  (idempoten).
Karena preprocessing sudah diterapkan, hasilannya harus BERBEDA.

  file idempoten : 0/8  (harus 0)

Output SUDAH diproses. Saat training, berikan langsung ke model
TANPA memanggil preprocess_input lagi.


## 8. Verifikasi Data Leakage

Preprocessing bersifat deterministik dan tidak menggunakan label. **Fold
dipertahankan apa adanya** dari `nodule_manifest.csv`.

In [13]:
# --- CELL 8 - LEAKAGE & KONSISTENSI FOLD ---------------------------------
src_man = pd.read_csv(MANIFEST_FILE, dtype={
    "PatientID": str, "SeriesInstanceUID": str, "NoduleClusterID": str})

key = ["PatientID", "SeriesInstanceUID", "NoduleClusterID"]
merged = (src_man[key + ["Label", "LabelBinary", "Fold"]]
          .merge(man[key + ["Label", "LabelBinary", "Fold"]],
                 on=key, suffixes=("_src", "_out")))

print("KONSISTENSI DENGAN MANIFEST SUMBER")
print("-" * 70)
print(f"  baris ter-join : {len(merged)} dari {len(src_man)}")
for col in ("Label", "LabelBinary", "Fold"):
    same = (merged[f"{col}_src"].astype(str)
            == merged[f"{col}_out"].astype(str)).all()
    print(f"  {col:<12} identik : {same}")
    assert same, f"{col} berubah antara manifest sumber dan output!"

print("\nLEAKAGE CHECK")
print("-" * 70)
total_overlap = 0
for f in range(1, 6):
    tr = set(man.loc[man["Fold"] != f, "PatientID"])
    va = set(man.loc[man["Fold"] == f, "PatientID"])
    ov = tr & va
    total_overlap += len(ov)
    print(f"  Fold {f}: overlap PatientID train/val = {len(ov)}")
n_multi = int((man.groupby("PatientID")["Fold"].nunique() > 1).sum())
print(f"  total overlap            : {total_overlap}")
print(f"  PatientID dengan >1 fold : {n_multi}")
print(f"  sample per fold          : "
      f"{man['Fold'].value_counts().sort_index().to_dict()}")
print(f"  label                    : {man['Label'].value_counts().to_dict()}")

assert total_overlap == 0, "Ada PatientID overlap train/val"
assert n_multi == 0, "Ada PatientID di lebih dari satu fold"
print("\nTidak ada leakage. Fold dipertahankan dari manifest sumber.")


KONSISTENSI DENGAN MANIFEST SUMBER
----------------------------------------------------------------------
  baris ter-join : 524 dari 524
  Label        identik : True
  LabelBinary  identik : True
  Fold         identik : True

LEAKAGE CHECK
----------------------------------------------------------------------
  Fold 1: overlap PatientID train/val = 0
  Fold 2: overlap PatientID train/val = 0
  Fold 3: overlap PatientID train/val = 0
  Fold 4: overlap PatientID train/val = 0
  Fold 5: overlap PatientID train/val = 0
  total overlap            : 0
  PatientID dengan >1 fold : 0
  sample per fold          : {1: 82, 2: 126, 3: 106, 4: 97, 5: 113}
  label                    : {'Benign': 381, 'Malignant': 143}

Tidak ada leakage. Fold dipertahankan dari manifest sumber.


## 9. Visual QC

Beberapa sample representatif saja. Figure disimpan ke
`results/preprocessing_qc/` dan **tidak** di-embed sebagai output notebook
agar notebook tetap ringan.

In [14]:
# --- CELL 9 - VISUAL QC (representatif, disimpan ke disk) ----------------
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# pilih sample yang mencakup variasi label dan ukuran nodul
cand = man.copy()
cand["core_px"] = cand["CropHeight"].astype(float)
variasi = []
for lab in ("Benign", "Malignant"):
    g = cand[cand["Label"] == lab]
    for q in (0.05, 0.50, 0.95):
        idx = g["core_px"].quantile(q)
        variasi.append((g["core_px"] - idx).abs().idxmin())
variasi = list(dict.fromkeys(variasi))[:6]
visual = man.loc[variasi]

fig, axes = plt.subplots(2, len(visual),
                         figsize=(3.2 * len(visual), 7.2))
axes = np.atleast_2d(axes)
for j, (_, r) in enumerate(visual.iterrows()):
    s = audit.CtSeries(r["SeriesInstanceUID"],
                       series_index[r["SeriesInstanceUID"]])
    mr, mc = audit.margin_px_pair_from_mm(
        r["MarginMM"], s.row_spacing, s.col_spacing)
    box = audit.build_crop_box(
        src_man[(src_man["PatientID"] == r["PatientID"])
                & (src_man["NoduleClusterID"] == r["NoduleClusterID"])].iloc[0],
        s, mr, mc)
    hu = s.read_hu(int(r["SelectedSliceIndex"]))

    ax = axes[0][j]
    ax.imshow(hu, cmap="gray", vmin=-1000, vmax=-400)
    ax.add_patch(Rectangle((box["c0"], box["r0"]),
                           box["c1"] - box["c0"], box["r1"] - box["r0"],
                           fill=False, ec="red", lw=1.3))
    ax.set_title(f"{r['Label']}\nk={r['SelectedSliceIndex']}", fontsize=7)
    ax.axis("off")

    ax = axes[1][j]
    from PIL import Image
    im = np.array(Image.open(PROJECT_ROOT / r["CropPath"]))
    ax.imshow(im, cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"crop 224 (up x{r['UpsampleFactor']:.1f})", fontsize=7)
    ax.axis("off")

fig.suptitle(f"Visual QC - margin {MARGIN_MM_FINAL:g} mm "
             f"(representatif, bukan seluruh dataset)", fontsize=10)
fig.tight_layout(rect=(0, 0, 1, 0.95))
qc_fig = RESULTS_DIR / "visual_qc_samples.png"
fig.savefig(qc_fig, dpi=100, bbox_inches="tight")
plt.close(fig)

print(f"Visual QC disimpan: {qc_fig.relative_to(PROJECT_ROOT)}")
print(f"Jumlah sample ditampilkan: {len(visual)} (bukan 524)")
print("Gambar TIDAK di-embed di notebook agar ukuran file tetap kecil.")
print("\nSample yang ditampilkan:")
display(visual[["PatientID", "NoduleClusterID", "Label", "Fold",
                "MarginMM", "CropHeight", "CropWidth",
                "UpsampleFactor", "SelectedSliceIndex"]].reset_index(drop=True))


Visual QC disimpan: results\preprocessing_qc\visual_qc_samples.png
Jumlah sample ditampilkan: 6 (bukan 524)
Gambar TIDAK di-embed di notebook agar ukuran file tetap kecil.

Sample yang ditampilkan:


,PatientID,NoduleClusterID,Label,Fold,MarginMM,CropHeight,CropWidth,UpsampleFactor,SelectedSliceIndex
0,LIDC-IDRI-0102,1.3.6.1.4.1.14519.5.2.1.6279.6001.234400932423...,Benign,1,5.0,21,20,10.667,42
1,LIDC-IDRI-0167,1.3.6.1.4.1.14519.5.2.1.6279.6001.336198008634...,Benign,1,5.0,26,26,8.615,60
2,LIDC-IDRI-0024,1.3.6.1.4.1.14519.5.2.1.6279.6001.206028343897...,Benign,2,5.0,37,34,6.054,84
3,LIDC-IDRI-0031,1.3.6.1.4.1.14519.5.2.1.6279.6001.124663713663...,Malignant,2,5.0,29,29,7.724,88
4,LIDC-IDRI-0217,1.3.6.1.4.1.14519.5.2.1.6279.6001.313283554967...,Malignant,2,5.0,41,38,5.463,34
5,LIDC-IDRI-0337,1.3.6.1.4.1.14519.5.2.1.6279.6001.266462563888...,Malignant,2,5.0,71,86,2.605,97


## 10. Final Report

In [15]:
# --- CELL 10 - LAPORAN AKHIR --------------------------------------------
cfg = json.loads((PREP_DIR / "preprocessing_config.json").read_text(encoding="utf-8"))

print("=" * 78)
print("LAPORAN PREPROCESSING DATASET LIDC-IDRI")
print("=" * 78)

print("\nINPUT")
print("-" * 78)
print(f"  sample total      : {len(df)}")
print(f"  Benign            : {int((df['LabelBinary'] == 0).sum())}")
print(f"  Malignant         : {int((df['LabelBinary'] == 1).sum())}")
print(f"  PatientID unik    : {df['PatientID'].nunique()}")
print(f"  SeriesInstanceUID : {df['SeriesInstanceUID'].nunique()}")
print(f"  NoduleClusterID   : {df['NoduleClusterID'].nunique()}")

print("\nMARGIN AUDIT")
print("-" * 78)
print(f"  kandidat diuji    : "
      f"{[f'{m:g} mm' for m in MARGIN_CANDIDATES_MM] + [f'{p} px' for p in MARGIN_CANDIDATES_PX]}")
print(f"  jumlah sample     : {df['NoduleClusterID'].nunique()} (bukan 1-2 contoh)")
print(f"  MARGIN FINAL      : {MARGIN_MM_FINAL} mm")
print(f"  alasan            : {MARGIN_DECISION_REASON}")
print(f"  dipilih dari      : audit geometris/preprocessing, "
      f"BUKAN performa model")

print("\nPREPROCESSING")
print("-" * 78)
print(f"  berhasil          : {len(man)}")
print(f"  gagal             : {0 if not len(man) else len(df) - len(man)}")
print(f"  margin            : {man['MarginMM'].unique()[0]} mm "
      f"(row {man['MarginRowPx'].min()}-{man['MarginRowPx'].max()} px, "
      f"col {man['MarginColPx'].min()}-{man['MarginColPx'].max()} px)")
print(f"  crop size median  : {man['CropHeight'].median():.0f} x "
      f"{man['CropWidth'].median():.0f} px")
print(f"  crop size rentang : {man['CropHeight'].min()} x "
      f"{man['CropWidth'].max()} px (terbesar)")
print(f"  output size       : {TARGET_SIZE[0]} x {TARGET_SIZE[1]} x {NUM_CHANNELS}")
print(f"  channel layout    : {CHANNEL_LAYOUT}")
print(f"  upsample median   : {man['UpsampleFactor'].median():.2f}")
print(f"  preprocessing     : {cfg['densenet_preprocessing']}")
print(f"  window            : [{cfg['window_hu_min']}, {cfg['window_hu_max']}] HU "
      f"(center {cfg['window_center_hu']}, width {cfg['window_width_hu']})")
print(f"  slice selection   : {cfg['slice_selection']}")
print(f"  koordinat         : {cfg['coordinate_convention']}")

print("\nQC")
print("-" * 78)
print(f"  NaN               : {int(man['HasNaN'].sum())}")
print(f"  Inf               : {int(man['HasInf'].sum())}")
print(f"  invalid crop      : {int(man['IsEmptyCrop'].sum())}")
print(f"  duplicate output  : "
      f"{int(man['NoduleClusterID'].duplicated().sum())}")
print(f"  missing output    : {len(df) - len(man)}")
print(f"  clipping          : {int(man['Clipped'].sum())}")
print(f"  leakage/fold change: 0 (fold identik manifest sumber)")
print(f"  pemeriksaan QC    : {len(qc) - n_bad}/{len(qc)} OK")

print("\nOUTPUT")
print("-" * 78)
for p in (PREP_DIR / "preprocessed_manifest.csv",
          PREP_DIR / "preprocessing_config.json",
          PREP_DIR / "preprocessing_qc.csv",
          RESULTS_DIR / "visual_qc_samples.png"):
    if p.exists():
        print(f"  {p.relative_to(PROJECT_ROOT)}  ({p.stat().st_size:,} bytes)")
print(f"  {CROP_RELDIR}/                       "
      f"({len(list(CROP_DIR.glob('*.png')))} file PNG)")
print(f"  {PREP_RELDIR}/                      "
      f"({len(list(PREP_DIR.glob('*.npy')))} file .npy)")

print("\nCATATAN PENTING UNTUK TAHAP TRAINING")
print("-" * 78)
print("  File .npy di dataset/04_preprocessed/ SUDAH berupa output")
print("  densenet.preprocess_input. Berikan LANGSUNG ke model.")
print("  Jangan memanggil preprocess_input lagi (double preprocessing).")


LAPORAN PREPROCESSING DATASET LIDC-IDRI

INPUT
------------------------------------------------------------------------------
  sample total      : 524
  Benign            : 381
  Malignant         : 143
  PatientID unik    : 225
  SeriesInstanceUID : 225
  NoduleClusterID   : 524

MARGIN AUDIT
------------------------------------------------------------------------------
  kandidat diuji    : ['0 mm', '3 mm', '5 mm', '8 mm', '0 px', '16 px', '32 px']
  jumlah sample     : 524 (bukan 1-2 contoh)
  MARGIN FINAL      : 5.0 mm
  alasan            : Nodul median 45% dari luas crop; upsample median x7.7 ke 224; hanya 5.9% sample butuh >=10x upsample (dibanding 47.7% pada 3 mm dan 79.0% pada 0 mm). Konteks parenkim dan pembuluh darah sekitar nodul tetap terlihat. Clipping 0% karena jarak minimum bbox ke tepi FOV 21.9 mm.
  dipilih dari      : audit geometris/preprocessing, BUKAN performa model

PREPROCESSING
------------------------------------------------------------------------------
  ber